In [0]:



# COMMAND ----------
# Cria o schema da camada Gold
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# COMMAND ----------
# Importações usadas nas agregações
from pyspark.sql.functions import *

# COMMAND ----------
# Tabela comercial agregada por ano, mês e categoria
df_vendas_comercial = (
    spark.table("silver.fat_pedido_total").alias("pt")
    .join(spark.table("silver.fat_itens_pedidos").alias("it"), "id_pedido", "left")
    .join(spark.table("silver.dim_produtos").alias("pr"), "id_produto", "left")
    .groupBy(
        year("data_pedido").alias("ano_venda"),
        month("data_pedido").alias("mes_venda"),
        col("pr.categoria_produto")
    )
    .agg(
        countDistinct("pt.id_pedido").alias("total_pedidos"),
        count("it.id_item").alias("qtd_itens_vendidos"),
        round(sum("pt.valor_total_pago_brl"), 2).alias("receita_total_brl"),
        round(sum("pt.valor_total_pago_usd"), 2).alias("receita_total_usd"),
        round(avg("pt.valor_total_pago_brl"), 2).alias("ticket_medio_brl")
    )
)

(
    df_vendas_comercial.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fat_vendas_comercial")
)

# COMMAND ----------
# Ranking dos 5 produtos mais vendidos
top_mais_vendidos = (
    spark.table("silver.fat_itens_pedidos").alias("it")
    .join(spark.table("silver.dim_produtos").alias("pr"), "id_produto", "left")
    .groupBy("pr.nome_produto", "pr.categoria_produto")
    .agg(count("*").alias("quantidade_vendida"))
    .orderBy(col("quantidade_vendida").desc())
    .limit(5)
)

display(top_mais_vendidos)

# COMMAND ----------
# Ranking dos 5 produtos menos vendidos
top_menos_vendidos = (
    spark.table("silver.fat_itens_pedidos").alias("it")
    .join(spark.table("silver.dim_produtos").alias("pr"), "id_produto", "left")
    .groupBy("pr.nome_produto", "pr.categoria_produto")
    .agg(count("*").alias("quantidade_vendida"))
    .orderBy(col("quantidade_vendida").asc(), col("pr.nome_produto").asc())
    .limit(5)
)

display(top_menos_vendidos)

# COMMAND ----------
# Tabela agregada de satisfação de clientes por categoria, vendedor e estado
df_avaliacoes_clientes = (
    spark.table("silver.fat_avaliacoes_pedidos").alias("av")
    .join(spark.table("silver.fat_itens_pedidos").alias("it"), "id_pedido", "left")
    .join(spark.table("silver.dim_produtos").alias("pr"), "id_produto", "left")
    .join(spark.table("silver.dim_vendedores").alias("vd"), "id_vendedor", "left")
    .groupBy(
        col("pr.categoria_produto"),
        col("vd.nome_vendedor"),
        col("vd.estado")
    )
    .agg(
        count("*").alias("total_avaliacoes"),
        round(avg("av.nota_avaliacao"), 2).alias("avaliacao_media"),
        sum(when(col("av.nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        sum(when(col("av.nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
    )
    .withColumn(
        "percentual_satisfacao",
        round((col("total_avaliacoes_positivas") / col("total_avaliacoes")) * 100, 2)
    )
)

(
    df_avaliacoes_clientes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fat_avaliacoes_clientes")
)

# COMMAND ----------
# Produto mais bem avaliado com desempate por volume de avaliações
produto_mais_bem_avaliado = (
    spark.table("silver.fat_avaliacoes_pedidos").alias("av")
    .join(spark.table("silver.fat_itens_pedidos").alias("it"), "id_pedido", "left")
    .join(spark.table("silver.dim_produtos").alias("pr"), "id_produto", "left")
    .groupBy("pr.nome_produto")
    .agg(
        round(avg("av.nota_avaliacao"), 2).alias("nota_media"),
        count("*").alias("volume_avaliacoes")
    )
    .orderBy(col("nota_media").desc(), col("volume_avaliacoes").desc())
    .limit(1)
)

display(produto_mais_bem_avaliado)

# COMMAND ----------
# Produto menos bem avaliado com desempate por volume de avaliações
produto_menos_bem_avaliado = (
    spark.table("silver.fat_avaliacoes_pedidos").alias("av")
    .join(spark.table("silver.fat_itens_pedidos").alias("it"), "id_pedido", "left")
    .join(spark.table("silver.dim_produtos").alias("pr"), "id_produto", "left")
    .groupBy("pr.nome_produto")
    .agg(
        round(avg("av.nota_avaliacao"), 2).alias("nota_media"),
        count("*").alias("volume_avaliacoes")
    )
    .orderBy(col("nota_media").asc(), col("volume_avaliacoes").desc())
    .limit(1)
)

display(produto_menos_bem_avaliado)

# COMMAND ----------
# Vendedor mais bem avaliado com desempate por volume de avaliações
vendedor_mais_bem_avaliado = (
    spark.table("silver.fat_avaliacoes_pedidos").alias("av")
    .join(spark.table("silver.fat_itens_pedidos").alias("it"), "id_pedido", "left")
    .join(spark.table("silver.dim_vendedores").alias("vd"), "id_vendedor", "left")
    .groupBy("vd.nome_vendedor")
    .agg(
        round(avg("av.nota_avaliacao"), 2).alias("nota_media"),
        count("*").alias("volume_avaliacoes")
    )
    .orderBy(col("nota_media").desc(), col("volume_avaliacoes").desc())
    .limit(1)
)

display(vendedor_mais_bem_avaliado)

# COMMAND ----------
# Vendedor menos bem avaliado com desempate por volume de avaliações
vendedor_menos_bem_avaliado = (
    spark.table("silver.fat_avaliacoes_pedidos").alias("av")
    .join(spark.table("silver.fat_itens_pedidos").alias("it"), "id_pedido", "left")
    .join(spark.table("silver.dim_vendedores").alias("vd"), "id_vendedor", "left")
    .groupBy("vd.nome_vendedor")
    .agg(
        round(avg("av.nota_avaliacao"), 2).alias("nota_media"),
        count("*").alias("volume_avaliacoes")
    )
    .orderBy(col("nota_media").asc(), col("volume_avaliacoes").desc())
    .limit(1)
)

display(vendedor_menos_bem_avaliado)

# COMMAND ----------
# Lista as tabelas criadas na camada Gold
spark.sql("SHOW TABLES IN gold").display()